In [ ]:
# Example 1: parse_gitignore from a temp file and match basic patterns.
import tempfile
from pathlib import Path
from gitignore_parser import parse_gitignore

tmp = Path(tempfile.mkdtemp())
gi = tmp / '.gitignore'
gi.write_text('*.pyc\n__pycache__/\nbuild/\n')

matches = parse_gitignore(gi)

assert matches(str(tmp / 'foo.pyc')) is True
assert matches(str(tmp / 'src' / 'bar.pyc')) is True
assert matches(str(tmp / 'foo.py')) is False
assert matches(str(tmp / 'build' / 'output.txt')) is True
print('Basic patterns matched correctly')


In [ ]:
# Example 2: Negation patterns (! prefix re-includes a file).
import tempfile
from pathlib import Path
from gitignore_parser import parse_gitignore

tmp = Path(tempfile.mkdtemp())
gi = tmp / '.gitignore'
gi.write_text('*.log\n!important.log\n')

matches = parse_gitignore(gi)

assert matches(str(tmp / 'debug.log')) is True
assert matches(str(tmp / 'important.log')) is False
assert matches(str(tmp / 'subdir' / 'other.log')) is True
print('Negation pattern re-included important.log as expected')


In [ ]:
# Example 3: Anchored vs. non-anchored patterns.
import tempfile
from pathlib import Path
from gitignore_parser import parse_gitignore

tmp = Path(tempfile.mkdtemp())
gi = tmp / '.gitignore'
# Leading slash anchors to gitignore's directory; bare name matches anywhere.
gi.write_text('/root_only.txt\nanywhere.txt\n')

matches = parse_gitignore(gi)

assert matches(str(tmp / 'root_only.txt')) is True
assert matches(str(tmp / 'sub' / 'root_only.txt')) is False
assert matches(str(tmp / 'anywhere.txt')) is True
assert matches(str(tmp / 'deep' / 'nested' / 'anywhere.txt')) is True
print('Anchored / non-anchored semantics correct')


In [ ]:
# Example 4: Directory-only patterns (trailing slash).
import os
import tempfile
from pathlib import Path
from gitignore_parser import parse_gitignore

tmp = Path(tempfile.mkdtemp())
gi = tmp / '.gitignore'
gi.write_text('cache/\n')

# Create actual filesystem entries so directory detection works.
(tmp / 'cache').mkdir()
(tmp / 'cache' / 'data.bin').write_text('x')
(tmp / 'cache_file').write_text('not a dir')

matches = parse_gitignore(gi)

assert matches(str(tmp / 'cache')) is True
assert matches(str(tmp / 'cache' / 'data.bin')) is True
# A file with name starting with 'cache' is not a directory and should NOT match 'cache/'.
assert matches(str(tmp / 'cache_file')) is False
print('Directory-only pattern matched directories, not similarly-named files')


In [ ]:
# Example 5: parse_gitignore from a string source via the file path-based API,
# combining wildcards, double-star globs, and a comment.
import tempfile
from pathlib import Path
from gitignore_parser import parse_gitignore

tmp = Path(tempfile.mkdtemp())
gi = tmp / '.gitignore'
gi.write_text(
    '# build artifacts\n'
    '**/dist/\n'
    'node_modules/\n'
    '*.tmp\n'
)

# Materialize directories so trailing-slash patterns can resolve.
(tmp / 'pkg' / 'dist').mkdir(parents=True)
(tmp / 'pkg' / 'dist' / 'bundle.js').write_text('x')
(tmp / 'node_modules').mkdir()

matches = parse_gitignore(gi)

results = {
    'pkg/dist/bundle.js': matches(str(tmp / 'pkg' / 'dist' / 'bundle.js')),
    'node_modules': matches(str(tmp / 'node_modules')),
    'scratch.tmp': matches(str(tmp / 'scratch.tmp')),
    'src/main.py': matches(str(tmp / 'src' / 'main.py')),
}
print(results)

assert results['pkg/dist/bundle.js'] is True
assert results['node_modules'] is True
assert results['scratch.tmp'] is True
assert results['src/main.py'] is False
print('\nAll gitignore_parser smoke tests passed')
